# gramlab — chat
Ask the retrieval-conditioned 25M model questions. **Interactive use only** (widgets don't run in Save & Run All).
Setup once: *Add Input → Your Work → Notebooks → gramlab* (latest version) and, for the BM25 + meaning-map hybrid retriever,
*gramlab-map* (latest); **Accelerator: None** (CPU is enough). Then: **Run All**, wait ~40 s, type in the box.

Main function: `ask(question)` → answer. `ask(question, retriever='bm25')` or `'hybrid'` to choose; `k=2` for two passages.

In [ ]:
import os, sys, glob, subprocess, json, numpy as np, torch, ipywidgets as w
from IPython.display import display
if not os.path.exists('/kaggle/working/gramlab'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/furevych/gramlab.git', '/kaggle/working/gramlab'], check=True)
sys.path.insert(0, '/kaggle/working/gramlab')
cands = sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki_rag/ckpt.pt', recursive=True),
               key=lambda p: -torch.load(p, map_location='cpu', weights_only=False)['step'])
if not cands:
    raise SystemExit('Add the gramlab notebook output as an input (right panel -> Input -> Add Input -> Your Work)')
os.chdir(cands[0].split('/work/')[0])
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
model, tok, state = load_checkpoint('work/runs/small_wiki_rag/ckpt.pt', 'cpu')
index = BM25Index.load('work/bm25')
print(f"writer step {state['step']}, BM25 over {len(index):,} passages, from {os.getcwd()}")

# optional: the meaning map (gramlab-map output) for a BM25 + map hybrid retriever
MAP = None
m2 = glob.glob('/kaggle/input/**/results/map2.json', recursive=True)
if m2:
    from gramlab.map import MapEncoder
    mroot = m2[0].split('/results/')[0]
    best = json.load(open(m2[0]))['best_for_hybrid']
    MAP = MapEncoder(f'{mroot}/work/map2/{best}')
    P = np.load(f'{mroot}/work/map2/{best}/passages.npy').astype(np.float32)
    try:
        import spacy; NLP = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'lemmatizer']) if MAP.use_slots else None
    except Exception:
        NLP = None
    print(f'meaning map loaded ({best}); passages {P.shape}; default retriever = hybrid')
DEFAULT = 'hybrid' if MAP is not None else 'bm25'

def tag(q):
    if NLP is None: return (q.split(), ['NOUN'] * len(q.split()))
    d = NLP(q); return ([t.text for t in d], [t.pos_ for t in d])

def retrieve(q, k=1, retriever=None):
    retriever = retriever or DEFAULT
    if retriever == 'hybrid' and MAP is not None:
        hits = index.search(q, 50)
        if not hits: return []
        qv = MAP.encode([tag(q)])[0]; mx = max(h[0] for h in hits)
        sc = sorted(((h[0] / mx + float(qv @ P[h[1]]), h[1]) for h in hits), reverse=True)[:k]
        out = []
        for s, i in sc:
            p = index.passage(i); p['score'] = round(s, 3); p['text'] = ' '.join(p['text'].split()[:90]); out.append(p)
        return out
    return index.retrieve(q, k=k, max_words=90)

def ask(q, k=1, retriever=None, show_passage=True, temperature=0.3):
    p = retrieve(q, k, retriever) or [{'title': '(no match)', 'text': 'No passage found.', 'score': 0}]
    ids = [USER] + tok.encode(format_rag_user(p, q)) + [END, ASSISTANT]
    out = model.generate(torch.tensor([ids[-(model.ctx - 1):]]), max_new_tokens=80, temperature=temperature,
                         top_k=40, top_p=0.9, repetition_penalty=1.2, stop_ids=(END, EOS))
    if show_passage:
        for x in p: print(f"  [read: {x['title']}]")
    return tok.decode(out[0, len(ids):].tolist())

box = w.Text(placeholder='ask about a place, a person, a thing...', layout=w.Layout(width='70%'), continuous_update=False)
btn, out = w.Button(description='Send'), w.Output()
def on_send(_=None):
    q = box.value.strip()
    if not q: return
    box.value = ''
    with out:
        print('you>  ', q); print('model>', ask(q)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

In [ ]:
# the main function, without the widget — one question per cell
ask("Who was Winston Churchill?")